# 04 · Robustness

Map the failure envelope of the pipeline you picked. The goal is not a single accuracy number but knowing **when to distrust a fix** — a navigation system that reports a wrong position confidently is worse than one that reports nothing.

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Flat src/ is installed editable, so these import from anywhere.
# autoreload 2 above means editing src/*.py takes effect on the next cell
# run — no kernel restart.
import matching
import simulation
import metrics

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"


def show(img, title=None, figsize=(11, 7)):
    """Inline display for OpenCV arrays (handles BGR->RGB and grayscale)."""
    plt.figure(figsize=figsize)
    gray = img.ndim == 2
    plt.imshow(img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
               cmap="gray" if gray else None)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("ready ·", cv2.__version__)


## Degradations, one axis at a time

Rotation, altitude/scale, brightness and contrast, motion blur, sensor noise, JPEG artifacts, partial occlusion (cloud). One variable per sweep, error curve per variable.

## Terrain dependence

Re-run across the region types from notebook 01. Expect water and farmland to fail where city centres succeed easily.

## The negative test

Feed frames cropped from a region that is **not** in the reference map and confirm the pipeline reports no fix instead of inventing one. Most people skip this; it is the test that decides whether the system is usable.

## Calibrate confidence

Plot your confidence signal (inlier count/ratio) against actual error. A usable signal separates good fixes from bad ones — if they overlap, you need a better signal before trusting any threshold. Pick the threshold from this plot, then report what fraction of frames you reject and how accurate the survivors are.